In [51]:
import pandas as pd
import numpy as np

In [52]:
df = pd.read_csv(r'C:\Users\Junayed\pandas_prac\Aug_4\HR\messy_hr_day4.csv')

In [53]:
df.head(5)

,EmployeeID,FullName,Email,Department,Country,HireDate,TerminationDate,Status,Salary,Currency,Bonus,TenureYears,PerformanceRating,ReviewDate,Notes
0,EMP001,Wendy Sato,wendy.sato@corp.com,Engineering,USA,2019-03-01,NaN,Active,$92000,USD,$4500,5,85%,2024-02-15,NaN
1,EMP002,Bram Coetzee,bram.coetzee@corp.com,marketing,UK,2020-06-15,NaN,Active,£58000,GBP,£2200,-999,0.78,2024-01-20,NaN
2,EMP003,Ines Duarte,ines.duarte@corp.com,Sales,Germany,2021-01-10,2023-11-01,Terminated,€61000,EUR,€1800,3,72,2023-10-01,NaN
3,EMP004,Otto Kessler,otto.kessler@corp.com,Engineering,Germany,2018-09-05,NaN,Active,€88000,EUR,€5000,6,91%,2018-05-01,Review before hire?
4,EMP005,Wendy Sato,wendy.sato@corp.com,Engineering,USA,2019-03-01,NaN,Active,$92000,USD,$4500,5,85%,2024-02-15,NaN


In [54]:
df.shape

(30, 15)

In [55]:
df.dtypes

EmployeeID             str
FullName               str
Email                  str
Department             str
Country                str
HireDate               str
TerminationDate        str
Status                 str
Salary                 str
Currency               str
Bonus                  str
TenureYears          int64
PerformanceRating      str
ReviewDate             str
Notes                  str
dtype: object

### Data Cleaning: Title Standardization
* **Issue:** The `Department` column contains mixed casing for non-standard entries (e.g., `marketing`, `Markting`).
* **Fix:** Standardize all instances to title case (`Markting`) to maintain consistent string formatting across the dataset.

In [56]:
dept_map = {
    "engineering": "Engineering", "enginering": "Engineering",
    "marketing": "Marketing", "markting": "Marketing",
    "sales": "Sales",
}
df["Department"] = df["Department"].str.strip().str.lower().map(dept_map)
df["Department"].value_counts()

Department
Engineering    13
Sales          10
Marketing       7
Name: count, dtype: int64

### Standardizing Missing Values:
* **Issue:** The `TerminationDate` column contains empty strings/blank values.
* **Fix:** Convert empty cells to explicit `np.nan` missing indicators to ensure consistent missing value handling across the dataset.

In [57]:
df["TerminationDate"] = df["TerminationDate"].replace("",np.nan)
df["TerminationDate"]

0            NaN
1            NaN
2     2023-11-01
3            NaN
4            NaN
5            NaN
6            NaN
7            NaN
8     2022-12-01
9            NaN
10           NaN
11           NaN
12           NaN
13           NaN
14           NaN
15           NaN
16    2021-01-01
17           NaN
18    2023-08-20
19           NaN
20           NaN
21           NaN
22           NaN
23    2021-09-01
24           NaN
25           NaN
26           NaN
27           NaN
28    2023-03-01
29           NaN
Name: TerminationDate, dtype: str

> **Salary Standardization:** 
> Parsed concatenated currency, Bonus strings, cleaned number formatting, and converted multi-currency salary values (`$`, `£`, `€`, `¥`, `₹`) to a unified **USD** baseline using standardized FX rates.

In [58]:
import re
rate_to_usd = {"USD": 1.0, "EUR": 1.08, "GBP": 1.27, "JPY": 0.0067, "INR": 0.012}

def to_usd(amount, currency):
    num = re.sub(r"[^\d.]", "", str(amount))
    return round(float(num) * rate_to_usd[currency], 2)

df["Salary_USD"] = df.apply(lambda r: to_usd(r["Salary"], r["Currency"]), axis=1)

df["Bonus_USD"] = df.apply(lambda r: to_usd(r["Bonus"], r["Currency"]), axis=1)


df[["Salary", "Currency", "Salary_USD", "Bonus", "Bonus_USD"]]

,Salary,Currency,Salary_USD,Bonus,Bonus_USD
0,$92000,USD,92000.0,$4500,4500.0
1,£58000,GBP,73660.0,£2200,2794.0
2,€61000,EUR,65880.0,€1800,1944.0
3,€88000,EUR,95040.0,€5000,5400.0
4,$92000,USD,92000.0,$4500,4500.0
5,$68000,USD,68000.0,$1500,1500.0
6,€54000,EUR,58320.0,€1200,1296.0
7,$76000,USD,76000.0,$3000,3000.0
8,€59000,EUR,63720.0,€1000,1080.0
9,$71000,USD,71000.0,$2000,2000.0


> Inserting the 'Salary_USD' & 'Bonus_USD' beside their own columns.

In [59]:
target_col = df.columns.get_loc("Salary")
df.insert(target_col+1, "Salary_USD", df.pop("Salary_USD"))

bonus_col = df.columns.get_loc("Bonus")
df.insert(bonus_col+1, "Bonus_USD", df.pop("Bonus_USD"))

df.head(5)


,EmployeeID,FullName,Email,Department,Country,HireDate,TerminationDate,Status,Salary,Salary_USD,Currency,Bonus,Bonus_USD,TenureYears,PerformanceRating,ReviewDate,Notes
0,EMP001,Wendy Sato,wendy.sato@corp.com,Engineering,USA,2019-03-01,NaN,Active,$92000,92000.0,USD,$4500,4500.0,5,85%,2024-02-15,NaN
1,EMP002,Bram Coetzee,bram.coetzee@corp.com,Marketing,UK,2020-06-15,NaN,Active,£58000,73660.0,GBP,£2200,2794.0,-999,0.78,2024-01-20,NaN
2,EMP003,Ines Duarte,ines.duarte@corp.com,Sales,Germany,2021-01-10,2023-11-01,Terminated,€61000,65880.0,EUR,€1800,1944.0,3,72,2023-10-01,NaN
3,EMP004,Otto Kessler,otto.kessler@corp.com,Engineering,Germany,2018-09-05,NaN,Active,€88000,95040.0,EUR,€5000,5400.0,6,91%,2018-05-01,Review before hire?
4,EMP005,Wendy Sato,wendy.sato@corp.com,Engineering,USA,2019-03-01,NaN,Active,$92000,92000.0,USD,$4500,4500.0,5,85%,2024-02-15,NaN


- In 'TenureYears' there is tenure duration like -999 which is not valid or looks fake, replacing it with np.nan.

In [60]:
df["TenureYears"].value_counts()

TenureYears
 2      8
-999    6
 3      5
 1      4
 5      3
 4      2
 6      1
 7      1
Name: count, dtype: int64

In [61]:
df["TenureYears"] = df["TenureYears"].replace(-999, np.nan)
df["TenureYears"]

0     5.0
1     NaN
2     3.0
3     6.0
4     5.0
5     2.0
6     NaN
7     3.0
8     3.0
9     2.0
10    2.0
11    7.0
12    NaN
13    1.0
14    2.0
15    4.0
16    3.0
17    1.0
18    NaN
19    2.0
20    5.0
21    1.0
22    4.0
23    NaN
24    2.0
25    1.0
26    NaN
27    2.0
28    3.0
29    2.0
Name: TenureYears, dtype: float64

> **Rating Normalization:** 
> Converted mixed performance rating formats (decimals like `0.78`, whole numbers like `65`, percentages like `85%`, and `N/A` strings) into a unified **0%–100% scale** (handling `N/A` as `np.nan`).

In [62]:
df["PerformanceRating"].value_counts()

PerformanceRating
85%     2
0.68    2
0.78    1
72      1
91%     1
88%     1
65      1
79%     1
93%     1
70%     1
0.6     1
82%     1
77%     1
84%     1
71%     1
80%     1
86%     1
75%     1
0.83    1
68%     1
89%     1
73%     1
69%     1
81%     1
66%     1
87%     1
Name: count, dtype: int64

In [63]:
def clean_rating(val):
    val = str(val).strip().lower()

    if val in ("nan","n/a", ""):
        return np.nan
    if "%" in val:
        return val
    num = float(val)
    return num * 100 if num <=1 else (str(num) + "%")

df["PerformanceRating"] = df["PerformanceRating"].apply(clean_rating)

df["PerformanceRating"]

0       85%
1      78.0
2     72.0%
3       91%
4       85%
5      68.0
6       NaN
7       88%
8     65.0%
9       79%
10     68.0
11      93%
12      70%
13     60.0
14      82%
15      77%
16      84%
17      71%
18      NaN
19      80%
20      86%
21      75%
22     83.0
23      68%
24      89%
25      73%
26      69%
27      81%
28      66%
29      87%
Name: PerformanceRating, dtype: object

> HireDate / TeminationDate / ReviewDate

In [64]:
df["HireDate"] = pd.to_datetime(df["HireDate"]).dt.strftime("%Y-%m-%d")
df["TerminationDate"] = pd.to_datetime(df["TerminationDate"]).dt.strftime("%Y-%m-%d")
df["ReviewDate"] = pd.to_datetime(df["ReviewDate"]).dt.strftime("%Y-%m-%d")
df[["HireDate", "TerminationDate", "ReviewDate"]]

,HireDate,TerminationDate,ReviewDate
0,2019-03-01,NaN,2024-02-15
1,2020-06-15,NaN,2024-01-20
2,2021-01-10,2023-11-01,2023-10-01
3,2018-09-05,NaN,2018-05-01
4,2019-03-01,NaN,2024-02-15
5,2022-02-20,NaN,2024-03-01
6,2020-11-11,NaN,2023-08-15
7,2021-07-01,NaN,2024-01-10
8,2019-05-20,2022-12-01,2022-10-01
9,2022-08-15,NaN,2024-02-01


> Cross checking Employee status and termination date.

In [65]:
active_with_termdate = (df["Status"] == "Active") & (df["TerminationDate"].notna())

df.loc[active_with_termdate, ["EmployeeID", "Status", "TerminationDate"]]

,EmployeeID,Status,TerminationDate
16,EMP017,Active,2021-01-01
23,EMP024,Active,2021-09-01


> Two employee terminated but the status was not updated.

In [66]:
termination_emp = (df["Status"] == "Terminated") & (df["TerminationDate"].isna())

df.loc[termination_emp, ["EmployeeID", "Status", "TerminationDate", "Notes"]]

,EmployeeID,Status,TerminationDate,Notes
6,EMP007,Terminated,NaN,Missing termination date
12,EMP013,Terminated,NaN,Missing termination date
26,EMP027,Terminated,NaN,Missing termination date


In [67]:
df.loc[active_with_termdate, "Status"] = "Terminated"

df["NeedsReview"] = termination_emp

df[["EmployeeID", "Status", "TerminationDate", "NeedsReview"]].loc[df["NeedsReview"] | active_with_termdate]

,EmployeeID,Status,TerminationDate,NeedsReview
6,EMP007,Terminated,NaN,True
12,EMP013,Terminated,NaN,True
16,EMP017,Terminated,2021-01-01,False
23,EMP024,Terminated,2021-09-01,False
26,EMP027,Terminated,NaN,True


> HireDate vs ReviewDate

In [68]:
review_before_hire = df["ReviewDate"] < df["HireDate"]

df.loc[review_before_hire, ["EmployeeID", "HireDate", "ReviewDate"]]

,EmployeeID,HireDate,ReviewDate
3,EMP004,2018-09-05,2018-05-01


**EMP004:** a review four months before the hire date is impossible — but unlike the Status/TerminationDate case, there's no second field here to cross-check against and correct with confidence. Is `HireDate` wrong, or `ReviewDate` wrong? Nothing in the row tells you which. This is a flag-it case, not a fix-it case.

In [69]:
df["NeedsReview"] = df["NeedsReview"] | review_before_hire

# Notes

In [70]:
df["Notes"] = df["Notes"].astype(str).str.strip()

df["Notes"] = df["Notes"].replace(["", "nan", "None"], np.nan)

df["Notes"]

0                                  NaN
1                                  NaN
2                                  NaN
3                  Review before hire?
4                                  NaN
5                                  NaN
6             Missing termination date
7                                  NaN
8                                  NaN
9                                  NaN
10                          Duplicate?
11                                 NaN
12            Missing termination date
13                                 NaN
14                  Typo in department
15                                 NaN
16    Active but has termination date?
17                                 NaN
18                Duplicate of EMP007?
19                                 NaN
20                                 NaN
21                                 NaN
22                                 NaN
23    Active but has termination date?
24                                 NaN
25                       

# Finally, we check the duplicates and remove them from the file.

In [71]:
dup_cols = ["FullName", "Email", "Department", "HireDate"]

df[df.duplicated(subset=dup_cols, keep=False)]

,EmployeeID,FullName,Email,Department,Country,HireDate,TerminationDate,Status,Salary,Salary_USD,Currency,Bonus,Bonus_USD,TenureYears,PerformanceRating,ReviewDate,Notes,NeedsReview
0,EMP001,Wendy Sato,wendy.sato@corp.com,Engineering,USA,2019-03-01,NaN,Active,$92000,92000.0,USD,$4500,4500.0,5.0,85%,2024-02-15,NaN,False
4,EMP005,Wendy Sato,wendy.sato@corp.com,Engineering,USA,2019-03-01,NaN,Active,$92000,92000.0,USD,$4500,4500.0,5.0,85%,2024-02-15,NaN,False
5,EMP006,Priya Chandran,priya.chandran@corp.com,Sales,USA,2022-02-20,NaN,Active,$68000,68000.0,USD,$1500,1500.0,2.0,68.0,2024-03-01,NaN,False
6,EMP007,Marco Villani,marco.villani@corp.com,Marketing,Italy,2020-11-11,NaN,Terminated,€54000,58320.0,EUR,€1200,1296.0,NaN,NaN,2023-08-15,Missing termination date,True
10,EMP011,Priya Chandran,priya.chandran@corp.com,Sales,USA,2022-02-20,NaN,Active,$68000,68000.0,USD,$1500,1500.0,2.0,68.0,2024-03-01,Duplicate?,False
18,EMP019,Marco Villani,marco.villani@corp.com,Marketing,Italy,2020-11-11,2023-08-20,Terminated,€54000,58320.0,EUR,€1200,1296.0,NaN,NaN,2023-08-15,Duplicate of EMP007?,False


In [72]:
df = df.drop_duplicates(subset=dup_cols, keep ='first').reset_index(drop=True)


In [74]:
df

,EmployeeID,FullName,Email,Department,Country,HireDate,TerminationDate,Status,Salary,Salary_USD,Currency,Bonus,Bonus_USD,TenureYears,PerformanceRating,ReviewDate,Notes,NeedsReview
0,EMP001,Wendy Sato,wendy.sato@corp.com,Engineering,USA,2019-03-01,NaN,Active,$92000,92000.0,USD,$4500,4500.0,5.0,85%,2024-02-15,NaN,False
1,EMP002,Bram Coetzee,bram.coetzee@corp.com,Marketing,UK,2020-06-15,NaN,Active,£58000,73660.0,GBP,£2200,2794.0,NaN,78.0,2024-01-20,NaN,False
2,EMP003,Ines Duarte,ines.duarte@corp.com,Sales,Germany,2021-01-10,2023-11-01,Terminated,€61000,65880.0,EUR,€1800,1944.0,3.0,72.0%,2023-10-01,NaN,False
3,EMP004,Otto Kessler,otto.kessler@corp.com,Engineering,Germany,2018-09-05,NaN,Active,€88000,95040.0,EUR,€5000,5400.0,6.0,91%,2018-05-01,Review before hire?,True
4,EMP006,Priya Chandran,priya.chandran@corp.com,Sales,USA,2022-02-20,NaN,Active,$68000,68000.0,USD,$1500,1500.0,2.0,68.0,2024-03-01,NaN,False
5,EMP007,Marco Villani,marco.villani@corp.com,Marketing,Italy,2020-11-11,NaN,Terminated,€54000,58320.0,EUR,€1200,1296.0,NaN,NaN,2023-08-15,Missing termination date,True
6,EMP008,Chiamaka Eze,chiamaka.eze@corp.com,Engineering,Nigeria,2021-07-01,NaN,Active,$76000,76000.0,USD,$3000,3000.0,3.0,88%,2024-01-10,NaN,False
7,EMP009,Felix Hartmann,felix.hartmann@corp.com,Sales,Germany,2019-05-20,2022-12-01,Terminated,€59000,63720.0,EUR,€1000,1080.0,3.0,65.0%,2022-10-01,NaN,False
8,EMP010,Nadia Popescu,nadia.popescu@corp.com,Engineering,Romania,2022-08-15,NaN,Active,$71000,71000.0,USD,$2000,2000.0,2.0,79%,2024-02-01,NaN,False
9,EMP012,Ravi Subramaniam,ravi.subramaniam@corp.com,Engineering,India,2017-04-10,NaN,Active,$95000,95000.0,USD,$6000,6000.0,7.0,93%,2024-01-25,NaN,False


In [75]:
df.shape

(27, 18)

In [76]:
df.dtypes

EmployeeID               str
FullName                 str
Email                    str
Department               str
Country                  str
HireDate                 str
TerminationDate          str
Status                   str
Salary                   str
Salary_USD           float64
Currency                 str
Bonus                    str
Bonus_USD            float64
TenureYears          float64
PerformanceRating     object
ReviewDate               str
Notes                    str
NeedsReview             bool
dtype: object

In [77]:
df[df["NeedsReview"]][["EmployeeID","Status","HireDate","TerminationDate","ReviewDate","NeedsReview"]]

,EmployeeID,Status,HireDate,TerminationDate,ReviewDate,NeedsReview
3,EMP004,Active,2018-09-05,NaN,2018-05-01,True
5,EMP007,Terminated,2020-11-11,NaN,2023-08-15,True
10,EMP013,Terminated,2020-03-01,NaN,2023-06-01,True
23,EMP027,Terminated,2020-10-15,NaN,2023-07-01,True


In [78]:
df.to_excel("Cleaned_HR_data.xlsx", index=False)

In [79]:
df.to_csv("Cleaned_HR_data.csv", index=False)